# Notebook 5: State Clustering via K-Means

This notebook clusters US states by their waste management profiles using K-Means,
then identifies priority targets for policy expansion (organic waste bans).

**Outputs:**
- `outputs/ml/charts/cluster_profiles.png`
- `outputs/ml/charts/cluster_map.png`
- `outputs/ml/charts/cluster_ban_overlay.png`
- `outputs/ml/cluster_results.json`
- `outputs/ml/_state_2024_clustered.parquet`

---
## 1. Setup & Data Loading

In [1]:
import os, sys
# Ensure we're at project root
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

In [2]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import json
import shutil
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

SECTOR_COLORS = {
    'Farm': '#2C5F2D',
    'Foodservice': '#F96167',
    'Manufacturing': '#065A82',
    'Residential': '#F9E795',
    'Retail': '#028090'
}

In [3]:
# Load enriched state data and policy info
state_df = pd.read_parquet('outputs/ml/_state_df_enriched.parquet')
policy = pd.read_csv('data/policy_timeline.csv')
ban_states = set(policy['state'].tolist())

print(f"State data shape: {state_df.shape}")
print(f"Ban states: {sorted(ban_states)}")

State data shape: (216930, 38)
Ban states: ['California', 'Connecticut', 'Maine', 'Maryland', 'Massachusetts', 'New Hampshire', 'New Jersey', 'New York', 'Oregon', 'Rhode Island', 'Vermont', 'Washington']


---
## 2. Build State Feature Vectors

In [4]:
# Aggregate to state level for 2024
state_2024 = state_df[state_df['year'] == 2024].groupby('state').agg(
    tons_surplus=('tons_surplus', 'sum'),
    tons_waste=('tons_waste', 'sum'),
    tons_landfill=('tons_landfill', 'sum'),
    tons_composting=('tons_composting', 'sum'),
    tons_donations=('tons_donations', 'sum'),
    tons_animal_feed=('tons_animal_feed', 'sum'),
    co2e=('surplus_total_100_year_mtco2e_footprint', 'sum'),
    us_dollars=('us_dollars_surplus', 'sum'),
    meals_wasted=('meals_wasted', 'sum')
).reset_index()

# Derived rates
state_2024['landfill_rate'] = state_2024['tons_landfill'] / state_2024['tons_waste']
state_2024['compost_rate'] = state_2024['tons_composting'] / state_2024['tons_waste']
state_2024['donation_rate'] = state_2024['tons_donations'] / state_2024['tons_waste']
state_2024['feed_rate'] = state_2024['tons_animal_feed'] / state_2024['tons_waste']

# Sector composition per state
for sector in ['Farm', 'Foodservice', 'Manufacturing', 'Residential', 'Retail']:
    sec_tons = state_df[(state_df['year']==2024) & (state_df['sector']==sector)].groupby('state')['tons_surplus'].sum()
    state_2024 = state_2024.merge(sec_tons.rename(f'pct_{sector.lower()}'), left_on='state', right_index=True, how='left')
    state_2024[f'pct_{sector.lower()}'] = state_2024[f'pct_{sector.lower()}'] / state_2024['tons_surplus']

# Ban indicator
state_2024['has_ban'] = state_2024['state'].isin(ban_states).astype(int)

print(f"State 2024 feature matrix: {state_2024.shape}")
state_2024.head(3)

State 2024 feature matrix: (50, 20)


,state,tons_surplus,tons_waste,tons_landfill,tons_composting,tons_donations,tons_animal_feed,co2e,us_dollars,meals_wasted,landfill_rate,compost_rate,donation_rate,feed_rate,pct_farm,pct_foodservice,pct_manufacturing,pct_residential,pct_retail,has_ban
0,Alabama,7.510602e+05,6.527950e+05,400187.867858,132751.120189,19025.954063,45268.572506,3.411149e+06,5.448215e+09,1.220057e+09,0.613038,0.203358,0.029145,0.069346,0.084120,0.257143,0.093521,0.471568,0.093648,0
1,Alaska,9.272249e+04,7.866108e+04,53320.728856,19242.234900,2743.249038,8565.283659,4.588715e+05,8.309915e+08,1.499654e+08,0.677854,0.244622,0.034874,0.108888,0.004667,0.238890,0.116814,0.533614,0.106015,0
2,Arizona,1.957735e+06,1.849348e+06,707055.912878,261961.273467,32764.211898,69522.790136,5.078913e+06,9.026390e+09,3.208284e+09,0.382327,0.141651,0.017717,0.037593,0.402159,0.148588,0.036626,0.344926,0.067702,0


---
## 3. K-Means Clustering

In [5]:
# Clustering features
cluster_features = ['landfill_rate', 'compost_rate', 'donation_rate', 'feed_rate',
                    'pct_farm', 'pct_foodservice', 'pct_residential', 'pct_retail']
X_cluster = state_2024[cluster_features].fillna(0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

# Elbow method with silhouette scores
inertias = []
sil_scores = []
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

best_k = sil_scores.index(max(sil_scores)) + 2
print(f"Silhouette scores: {dict(zip(range(2,8), [round(s,3) for s in sil_scores]))}")
print(f"Best k by silhouette: {best_k}")

Silhouette scores: {2: 0.413, 3: 0.41, 4: 0.237, 5: 0.282, 6: 0.269, 7: 0.283}
Best k by silhouette: 2


In [6]:
# Use k=4 for interpretability
k_use = 4
kmeans = KMeans(n_clusters=k_use, random_state=42, n_init=10)
state_2024['cluster'] = kmeans.fit_predict(X_scaled)

# Name clusters based on profiles
cluster_profiles = state_2024.groupby('cluster')[cluster_features].mean()
print("\nCluster profiles:")
print(cluster_profiles.round(3).to_string())

cluster_names = {}
for c in range(k_use):
    profile = cluster_profiles.loc[c]
    states_in = state_2024[state_2024['cluster'] == c]
    n_ban = states_in['has_ban'].sum()
    n_total = len(states_in)

    if profile['compost_rate'] >= cluster_profiles['compost_rate'].quantile(0.75):
        name = "Diversion Leaders"
    elif profile['landfill_rate'] >= cluster_profiles['landfill_rate'].quantile(0.75):
        name = "Landfill-Dependent"
    elif profile['pct_farm'] >= cluster_profiles['pct_farm'].quantile(0.75):
        name = "Agricultural States"
    elif profile['pct_residential'] >= cluster_profiles['pct_residential'].quantile(0.75):
        name = "Urban Consumer States"
    else:
        name = "Mixed Economy States"

    cluster_names[c] = name
    print(f"  Cluster {c} '{name}': {n_total} states ({n_ban} with bans)")
    print(f"    States: {', '.join(sorted(states_in['state'].tolist()))}")

state_2024['cluster_name'] = state_2024['cluster'].map(cluster_names)


Cluster profiles:
         landfill_rate  compost_rate  donation_rate  feed_rate  pct_farm  pct_foodservice  pct_residential  pct_retail
cluster                                                                                                               
0                0.207         0.118          0.042      0.201     0.462            0.087            0.159       0.031
1                0.539         0.247          0.031      0.092     0.056            0.260            0.489       0.096
2                0.507         0.313          0.056      0.521     0.076            0.120            0.260       0.051
3                0.434         0.198          0.032      0.131     0.153            0.182            0.374       0.074
  Cluster 0 'Agricultural States': 5 states (3 with bans)
    States: California, Idaho, Oregon, Washington, Wisconsin
  Cluster 1 'Landfill-Dependent': 22 states (6 with bans)
    States: Alabama, Alaska, Colorado, Connecticut, Hawaii, Kansas, Kentucky, Maryland, Ma

---
## 4. Cluster Visualization

In [7]:
# 1. Cluster profiles bar chart
fig, ax = plt.subplots(figsize=(14, 7))
display_features = ['landfill_rate', 'compost_rate', 'donation_rate',
                    'pct_farm', 'pct_foodservice', 'pct_residential', 'pct_retail']
display_labels = ['Landfill\nRate', 'Compost\nRate', 'Donation\nRate',
                  '% Farm', '% Foodservice', '% Residential', '% Retail']

x = np.arange(len(display_features))
width = 0.18
colors = ['#2C5F2D', '#F96167', '#065A82', '#028090']

for i, c in enumerate(range(k_use)):
    vals = cluster_profiles.loc[c, display_features].values
    ax.bar(x + i*width, vals, width, label=f"C{c}: {cluster_names[c]}", color=colors[i], alpha=0.85)

ax.set_xticks(x + width*(k_use-1)/2)
ax.set_xticklabels(display_labels, fontsize=10)
ax.set_ylabel('Rate / Proportion')
ax.set_title('State Cluster Profiles: Waste Management Characteristics', fontsize=14, fontweight='bold')
ax.legend(fontsize=9, loc='upper right')
ax.grid(axis='y', alpha=0.3)
plt.savefig('outputs/ml/charts/cluster_profiles.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print("Saved cluster_profiles.png")

Saved cluster_profiles.png


In [8]:
# 2. Cluster map — states by cluster & landfill volume
from matplotlib.patches import Patch

fig, ax = plt.subplots(figsize=(14, 12))
sorted_states = state_2024.sort_values(['cluster', 'tons_landfill'], ascending=[True, True])
colors_map = {c: colors[c] for c in range(k_use)}
bar_colors = [colors_map[c] for c in sorted_states['cluster']]
y_pos = range(len(sorted_states))
ax.barh(y_pos, sorted_states['tons_landfill'] / 1e6, color=bar_colors, edgecolor='white', linewidth=0.3)

for idx, (_, row) in enumerate(sorted_states.iterrows()):
    if row['has_ban']:
        ax.text(row['tons_landfill']/1e6 + 0.01, idx, '*', fontsize=10, color='red', va='center')

ax.set_yticks(y_pos)
ax.set_yticklabels(sorted_states['state'], fontsize=7)
ax.set_xlabel('Tons to Landfill (Millions)', fontsize=11)
ax.set_title('States by Cluster & Landfill Volume (* = Ban State)', fontsize=14, fontweight='bold')

legend_elements = [Patch(facecolor=colors[c], label=f"{cluster_names[c]}") for c in range(k_use)]
legend_elements.append(plt.Line2D([0],[0], marker='*', color='red', label='Ban State', linestyle='None', markersize=12))
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)
ax.grid(axis='x', alpha=0.3)
plt.savefig('outputs/ml/charts/cluster_map.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print("Saved cluster_map.png")

Saved cluster_map.png


In [9]:
# 3. Copy as ban overlay
shutil.copy('outputs/ml/charts/cluster_map.png', 'outputs/ml/charts/cluster_ban_overlay.png')
print("Saved cluster_ban_overlay.png")

Saved cluster_ban_overlay.png


---
## 5. Policy Targeting

In [10]:
# Identify priority targets: non-ban states in clusters that contain ban states
ban_clusters = state_2024[state_2024['has_ban']==1]['cluster'].unique()
priority_targets = state_2024[
    (state_2024['has_ban']==0) & (state_2024['cluster'].isin(ban_clusters))
].sort_values('tons_landfill', ascending=False)

print(f"Priority targets for ban expansion ({len(priority_targets)} states):")
print(priority_targets[['state','cluster_name','landfill_rate','tons_landfill']].to_string(index=False))

Priority targets for ban expansion (34 states):
         state         cluster_name  landfill_rate  tons_landfill
         Texas   Landfill-Dependent       0.643918   2.625717e+06
       Florida Mixed Economy States       0.368941   1.560799e+06
          Ohio   Landfill-Dependent       0.605507   1.018092e+06
       Georgia Mixed Economy States       0.536418   9.924507e+05
      Illinois Mixed Economy States       0.583199   9.808567e+05
North Carolina Mixed Economy States       0.507665   9.362976e+05
      Michigan Mixed Economy States       0.413666   7.183936e+05
       Arizona Mixed Economy States       0.382327   7.070559e+05
      Virginia   Landfill-Dependent       0.533928   6.878993e+05
     Tennessee   Landfill-Dependent       0.628245   6.578929e+05
  Pennsylvania Mixed Economy States       0.362516   6.452462e+05
      Colorado   Landfill-Dependent       0.643668   6.031812e+05
       Indiana Mixed Economy States       0.465148   5.108935e+05
      Missouri Mixed Economy

---
## 6. Save Results

In [11]:
# Save cluster_results.json
cluster_results = {
    "method": "K-Means",
    "n_clusters": k_use,
    "silhouette_scores": dict(zip([str(k) for k in range(2,8)], [round(s,4) for s in sil_scores])),
    "clusters": {},
    "priority_ban_targets": priority_targets['state'].tolist()
}

for c in range(k_use):
    states_in = state_2024[state_2024['cluster'] == c]
    cluster_results["clusters"][cluster_names[c]] = {
        "cluster_id": c,
        "n_states": len(states_in),
        "states": sorted(states_in['state'].tolist()),
        "ban_states": sorted(states_in[states_in['has_ban']==1]['state'].tolist()),
        "non_ban_states": sorted(states_in[states_in['has_ban']==0]['state'].tolist()),
        "avg_landfill_rate": round(states_in['landfill_rate'].mean(), 4),
        "avg_compost_rate": round(states_in['compost_rate'].mean(), 4),
        "avg_donation_rate": round(states_in['donation_rate'].mean(), 4),
        "total_tons_landfill": round(states_in['tons_landfill'].sum(), 0)
    }

with open('outputs/ml/cluster_results.json', 'w') as f:
    json.dump(cluster_results, f, indent=2)
print("Saved to outputs/ml/cluster_results.json")

Saved to outputs/ml/cluster_results.json


In [12]:
# Save clustered state data for downstream
state_2024.to_parquet('outputs/ml/_state_2024_clustered.parquet', index=False)
print("Saved to outputs/ml/_state_2024_clustered.parquet")

Saved to outputs/ml/_state_2024_clustered.parquet
